# Lesson 2 · Exploring countries with Python

**Question:** What can country-level data tell us about life expectancy, income, and population?

This is an **optional exercise**. You do not need any programming experience. The cells already contain working code; your job is to **run**, **change a few labeled settings**, and **observe what happens**.

### How to use this notebook
1. Click inside a code cell (the box containing Python code).
2. Press **Shift + Enter** to run it. The computer executes the instructions in that cell, and any result appears **underneath** it.
3. You can edit the code and run the cell again. Changes are not applied until you run it.
4. Work **from top to bottom** once. After changing a setting, rerun that cell **and the cells below it** that use the setting.
5. Don't worry about breaking anything; rerun the original code to start over. You can download a copy of your notebook from the File menu.

**Try this first:**

In [ ]:
print("Hello, world!")
print("I just ran my first Python cell.")

### Change something small

A cell can also **calculate**. Change `NUMBER_OF_PEOPLE` from `4` to `6`. Predict what the answer will be, then press **Shift + Enter**.

In [ ]:
NUMBER_OF_PEOPLE = 4       # CHANGE ME: try 6 or 10
APPLES_PER_PERSON = 3        # CHANGE ME: try 2

total_apples = NUMBER_OF_PEOPLE * APPLES_PER_PERSON
print("Total apples:", total_apples)

---
## 1. Load our country data

The dataset is in the accompanying file **`countries.csv`**. Every **row** represents **one country in one year**. Every **column** is something we know about it.

The code below loads the file, then **displays the first 10 rows**. This is the information you have to work with!

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

data = pd.read_csv("countries.csv")
print("Rows:", len(data), "| Countries:", data["country"].nunique())
display(data.head(10))

### What do the columns mean?

| Column | Meaning | Units / examples |
|:--|:--|:--|
| `country` | Country name | e.g. Poland, Japan |
| `year` | Year of the observation | 1987, 1997, 2007 |
| `region` | Continent | Africa, Americas, Asia, Europe, Oceania |
| `life_expectancy` | Estimated life expectancy at birth | Years |
| `gdp_per_capita` | Economic output per person, adjusted for international comparison | Dataset international-dollar measure |
| `population` | Estimated number of residents | People |

The dataset is **real but historical**: 142 countries in 1987, 1997, and 2007, from the bundled Gapminder dataset distributed by Plotly. **Ukraine is not included in this particular historical dataset.** It is **not current data**. Definitions and estimates have limitations, and these are country-level observations, not data on individual people.

**Think:** What information is missing that might help explain why life expectancy varies?

---
## 2. Choose the question you want to explore

This is the main **settings cell**. Change the values on the **right side of the equals sign**. Keep the quotes (`" "`) around text.

- `YEAR`: try `1987`, `1997`, or `2007`. **Prediction:** will typical life expectancy be higher in later years?
- `REGION`: try `"All"`, `"Europe"`, `"Asia"`, `"Africa"`, `"Americas"`, or `"Oceania"`. **Prediction:** how will narrowing the region change the graph?
- `X_VARIABLE` and `Y_VARIABLE`: choose from `"life_expectancy"`, `"gdp_per_capita"`, `"population"`. **Prediction:** what relationship do you expect to see?
- `LOG_X`: use `True` for a **logarithmic** horizontal axis (often useful for GDP or population); use `False` for a linear axis.

Start with the defaults. **After changing this cell, run it again and rerun the analysis cells below.**

In [ ]:
# ===== EDIT THESE SETTINGS =====
YEAR = 2007                     # Try 1987 or 1997
REGION = "All"                  # Try "Europe", "Asia", "Africa", "Americas", "Oceania"
X_VARIABLE = "gdp_per_capita"   # Try "population" or "life_expectancy"
Y_VARIABLE = "life_expectancy" # Try "gdp_per_capita" or "population"
LOG_X = True                    # Try False to see what changes
# ===============================

# The code below checks your choices and selects the rows to study.
valid_years = [1987, 1997, 2007]
valid_regions = ["All"] + sorted(data["region"].unique().tolist())
valid_variables = ["life_expectancy", "gdp_per_capita", "population"]
assert YEAR in valid_years, f"Choose YEAR from {valid_years}"
assert REGION in valid_regions, f"Choose REGION from {valid_regions}"
assert X_VARIABLE in valid_variables, f"Choose X_VARIABLE from {valid_variables}"
assert Y_VARIABLE in valid_variables, f"Choose Y_VARIABLE from {valid_variables}"
assert isinstance(LOG_X, bool), "Set LOG_X to True or False (without quotes)"

selected = data[data["year"] == YEAR].copy()
if REGION != "All":
    selected = selected[selected["region"] == REGION].copy()

print(f"Selected {len(selected)} countries in {YEAR} | region: {REGION}")
display(selected[["country", "region", "life_expectancy", "gdp_per_capita", "population"]].head(8))

## 3. Describe the data before graphing it

A good first step is to compute summaries. Here we'll calculate the **mean**, **median**, and **standard deviation** of life expectancy for your chosen countries.

**Before running:** do you expect the mean and median to be identical?

In [ ]:
values = selected["life_expectancy"].dropna()
print("Life expectancy in", YEAR, "(", REGION, ")")
print("Mean:", round(values.mean(), 1), "years")
print("Median:", round(values.median(), 1), "years")
print("Standard deviation:", round(values.std(), 1), "years")
print("Countries included:", len(values))

## 4. Turn the distribution into a picture

A **histogram** groups countries into life-expectancy ranges. The height of each bar is the **number of countries** in that range.

**Question:** is the distribution symmetric? Is there one common cluster or more than one? Try another `YEAR` or `REGION` and run this cell again.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(selected["life_expectancy"].dropna(), bins=12, edgecolor="white")
ax.set(xlabel="Life expectancy at birth (years)", ylabel="Number of countries",
       title=f"Life expectancy in {YEAR} — {REGION}")
plt.show()

## 5. Make a scatterplot

Each dot represents **one country**. A scatterplot can reveal a relationship between **two numerical variables**.

The two axis variables and the year are taken from the settings cell. Try **`LOG_X = False`** after the first run. Does the pattern become easier or harder to see?

**Question:** does this show that income *causes* longer life expectancy? Why or why not?

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for region, group in selected.groupby("region"):
    ax.scatter(group[X_VARIABLE], group[Y_VARIABLE], label=region, alpha=0.75, s=40)

if LOG_X:
    ax.set_xscale("log")
ax.set(xlabel=X_VARIABLE.replace("_", " ").title(),
       ylabel=Y_VARIABLE.replace("_", " ").title(),
       title=f"{Y_VARIABLE.replace('_',' ').title()} vs. {X_VARIABLE.replace('_',' ').title()} — {YEAR}")
if REGION == "All":
    ax.legend(title="Region", fontsize=9)
plt.show()

## 6. Follow one country over time

We have three different years for each country. Change `COUNTRY` to see another country's history.

**Prediction:** would you expect life expectancy to rise from 1987 to 2007 for every country?

In [ ]:
COUNTRY = "Poland"             # Try "Japan", "Brazil", "India", or "United States"

assert COUNTRY in data["country"].values, "Choose a country from the dataset"
country_data = data[data["country"] == COUNTRY].sort_values("year")
display(country_data[["year", "life_expectancy", "gdp_per_capita"]])

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(country_data["year"], country_data["life_expectancy"], marker="o", linewidth=2)
ax.set(xlabel="Year", ylabel="Life expectancy (years)",
       title=f"Life expectancy over time — {COUNTRY}")
ax.set_xticks([1987, 1997, 2007])
plt.show()

---
## 7. Your turn: make a small investigation

Try changing **at least two settings**, rerunning the relevant cells, and observing what happens.

1. **Prediction:** What did you expect when you changed the year, region, or variable?
2. **Observation:** What did you actually see? Mention a number or a pattern from a graph.
3. **Limitation:** What information might be missing? Why doesn't an association automatically prove causation?

If you want a challenge: compare the mean and median for Europe versus Africa in 2007, or look for a country whose life expectancy changed unexpectedly.

**What to submit (if requested):** a short paragraph describing your investigation, or your downloaded notebook. You do not have to submit anything to complete the optional exercise.

### Data credit and limitations
This exercise uses a subset of the Gapminder country data included in Plotly Express (`plotly.express.data.gapminder()`): 142 countries × 1987, 1997, 2007. The numbers are historical estimates and comparisons can be affected by missing variables, differing definitions, and national averages. For updated data and documentation, see [Gapminder Data](https://www.gapminder.org/data/). This notebook uses **no live data connection**; it reads the bundled CSV.